In [1]:
import pandas as pd

### Data

In [2]:
data = pd.read_csv('../data/raw/GSE33000_series_matrix.txt', sep='\t', comment='!')

In [3]:
print(f'Data shape: {data.shape}')
data.head()

Data shape: (39280, 625)


,ID_REF,GSM1423780,GSM1423781,GSM1423782,GSM1423783,GSM1423784,GSM1423785,GSM1423786,GSM1423787,GSM1423788,...,GSM1424394,GSM1424395,GSM1424396,GSM1424397,GSM1424398,GSM1424399,GSM1424400,GSM1424401,GSM1424402,GSM1424403
0,10019475365,-0.016184,0.027599,-0.089661,-0.020493,-0.051699,0.075485,-0.046606,0.111980,0.018505,...,0.0844,0.1040,0.2140,0.00285,0.0945,0.1220,-0.0258,0.0186,0.14300,0.1600
1,10019481149,-0.017105,-0.023574,0.044863,0.003438,-0.064108,-0.053226,-0.071823,-0.051699,-0.024906,...,0.0409,0.0761,0.0633,0.03090,0.0319,0.0390,0.0897,0.1160,-0.00317,0.0359
2,10019495284,0.077440,-0.058746,-0.120759,0.072692,-0.051002,-0.183919,-0.074997,-0.090709,0.049640,...,-0.1360,-0.0547,-0.1920,-0.11200,-0.1230,-0.1520,0.5390,0.2280,-0.02580,-0.0968
3,10019687586,0.033595,0.032914,-0.030928,-0.033051,-0.049156,-0.011204,0.072451,0.126576,0.078031,...,-0.0120,0.0533,-0.0416,-0.01530,0.0483,0.0127,0.0419,0.1290,0.00619,-0.0360
4,10019713746,-0.001819,0.032487,-0.015073,-0.010303,0.063070,0.113860,-0.055590,0.127236,0.056135,...,0.1140,0.0270,0.2590,0.10100,0.2000,0.1170,-0.0359,0.0833,0.19500,0.1530


### Metadata

In [4]:
metadata = {}
count = 0

with open('../data/raw/GSE33000_series_matrix.txt') as file:
    for line in file:
        # Get everything that is metadata
        if line.startswith('!Sample_'):
            line = line.replace('"', '').split('\t')
            metadata[count] = line
            count += 1

metadata = pd.DataFrame(metadata)
columns = metadata.iloc[0]

# Remove "!Sample_" from every column
metadata.columns = [col.split('!Sample_')[1] for col in columns]

# Remove "\n" from rows
metadata = metadata.replace(r'\n', '', regex=True)
metadata = metadata.drop(metadata.index[0])


In [5]:
data_id = pd.Series(data.columns[1:]).reset_index(drop=True)
metadata_id = metadata['geo_accession'].reset_index(drop=True)

assert data_id.equals(metadata_id)

In [6]:
columns_unique = metadata.columns.is_unique
if not columns_unique:
    print('Columns are not unique.')
    duplicated = metadata.loc[:, metadata.columns.duplicated(keep=False)].copy()

    # Remove duplicated columns
    metadata = metadata.drop(columns=duplicated.columns)

    len_duplicated = duplicated.shape[1]

    # Remove column name from each row
    for i in range(len_duplicated):
        # Split label and value
        splitted = duplicated.iloc[:, i].str.split(': ', n=1)
        if splitted.str[0].nunique() == 1:
            label = splitted.str[0].iloc[0].replace(' ', '_')
            label = f'{label}_{duplicated.columns[i].split("_")[1]}'

            values = splitted.str[1]
            metadata[label] = values
        else:
            print('Labels are not the same.')

Columns are not unique.


In [7]:
metadata.head()

,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,treatment_protocol_ch1,...,contact_zip/postal_code,contact_country,supplementary_file,data_row_count,tissue_ch1,sample_type_ch1,tissue_ch2,age_ch2,gender_ch2,disease_status_ch2
1,PFC_1,GSM1423780,Public on Jul 31 2014,Jul 01 2014,Jul 31 2014,RNA,2,HBTRC_PF_Pool_1,Homo sapiens,NA,...,10029,USA,NONE,39280,prefrontal cortex brain,reference,prefrontal cortex brain,67 yrs,female,Alzheimer's disease
2,PFC_2,GSM1423781,Public on Jul 31 2014,Jul 01 2014,Jul 31 2014,RNA,2,HBTRC_PF_Pool_2,Homo sapiens,NA,...,10029,USA,NONE,39280,prefrontal cortex brain,reference,prefrontal cortex brain,88 yrs,male,Alzheimer's disease
3,PFC_3,GSM1423782,Public on Jul 31 2014,Jul 01 2014,Jul 31 2014,RNA,2,HBTRC_PF_Pool_3,Homo sapiens,NA,...,10029,USA,NONE,39280,prefrontal cortex brain,reference,prefrontal cortex brain,62 yrs,male,Alzheimer's disease
4,PFC_4,GSM1423783,Public on Jul 31 2014,Jul 01 2014,Jul 31 2014,RNA,2,HBTRC_PF_Pool_4,Homo sapiens,NA,...,10029,USA,NONE,39280,prefrontal cortex brain,reference,prefrontal cortex brain,90 yrs,female,Alzheimer's disease
5,PFC_5,GSM1423784,Public on Jul 31 2014,Jul 01 2014,Jul 31 2014,RNA,2,HBTRC_PF_Pool_5,Homo sapiens,NA,...,10029,USA,NONE,39280,prefrontal cortex brain,reference,prefrontal cortex brain,90 yrs,female,Alzheimer's disease


In [8]:
metadata['disease_status_ch2'].value_counts()

disease_status_ch2
Alzheimer's disease     310
non-demented            157
Huntington's disease    157
Name: count, dtype: int64

### Save data

In [9]:
data.to_csv('../data/processed/data.csv', index=False)
metadata.to_csv('../data/processed/metadata.csv', index=False)